Inports

In [1]:
# pip install pyodbc

In [2]:
# pip install -U imbalanced-learn

In [3]:
# pip install -U scikit-learn

In [4]:
#pip install mlxtend  

In [5]:
import pandas as pd
#Ig
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest
# Chi square
from sklearn.feature_selection import chi2
import numpy as np
# SU
import math
from scipy.stats import entropy
from sklearn import metrics

In [6]:
class ModelResult:
    def __init__(self, level_measure,project,b,rank,model_selection,intance_reduction,model,precision,recall,f,accuracy,count_metric,count_row,count_bag):
        self.level_measure = level_measure
        self.project = project
        self.b = b
        self.rank = rank
        self.model_selection = model_selection
        self.intance_reduction = intance_reduction
        self.model = model
        self.precision = precision
        self.recall = recall
        self.f = f
        self.accuracy = accuracy
        self.count_metric = count_metric
        self.count_row = count_row
        self.count_bag = count_bag
class SelectMeasure:
    def __init__(self,level_measure,project,b,rank,cs_su,ig_su,rf_su,cs_cos,ig_cos,rf_cos):
        self.level_measure = level_measure
        self.project = project
        self.b = b
        self.rank = rank
        self.cs_su = cs_su
        self.ig_su = ig_su
        self.rf_su = rf_su
        self.cs_cos = cs_cos
        self.ig_cos = ig_cos
        self.rf_cos = rf_cos

In [7]:
def GetMetrics(ytest, ypred,name):
    return pd.DataFrame([
        round(metrics.precision_score(ytest, ypred, average='binary',labels=np.unique(ypred)), 2),
        round(metrics.recall_score(ytest, ypred, average='binary',labels=np.unique(ypred)), 2),
        round(metrics.f1_score(ytest, ypred, average='binary',labels=np.unique(ypred)), 2),
        round(metrics.balanced_accuracy_score(ytest, ypred), 2)
    ],
                    index=["pr","re","f","Ac"],
                    columns=[name])
 

In [8]:
def GetMetricsResult(result,name):
    return pd.DataFrame([
        round(result.iloc[0].mean(), 2),
        round(result.iloc[1].mean(), 2),
        round(result.iloc[2].mean(), 2),
        round(result.iloc[3].mean(), 2)
    ],
                    index=["pr","re","f","Ac"],
                    columns=[name])

In [9]:
_scoring = {'acc': 'accuracy',
           'prec_macro': 'precision_macro',
           'rec_micro': 'recall_macro',
           'f':'f1_macro'
           }
def GetMetricsCross(results,name):
    return pd.DataFrame([
        round(results['test_prec_macro'].mean(), 2),
        round(results['test_rec_micro'].mean(), 2),
        round(results['test_f'].mean(), 2),
        round(results['test_acc'].mean(), 2)
    ],
                    index=["pr","re","f","Ac"],
                    columns=[name])

In [10]:
import os
def FileList(directory,list=[]):
  for filename in os.listdir(directory):
      f = os.path.join(directory, filename)
      if os.path.isdir(f):
        FileList(f,list)
      # checking if it is a file
      if os.path.isfile(f) and os.path.splitext(f)[-1].lower() == '.csv':
        list.append(f)
  return np.array(list)

In [11]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix

def NaiveBayes(X_train,y_train,X_test,y_test,name,n_validation = 10):
    result = pd.DataFrame([])
    for i in range(n_validation):
      model = MultinomialNB().fit(X_train, y_train)
      y_pred = model.predict(X_test)
      result = pd.concat([result,GetMetrics(y_test, y_pred,name)],axis=1)
    return GetMetricsResult(result,name)

In [12]:
from sklearn.neighbors import KNeighborsClassifier
def KNN(X_train,y_train,X_test,y_test,name,n_validation = 10):
     result = pd.DataFrame([])
     for i in range(n_validation):
          knn = KNeighborsClassifier(n_neighbors=3)
          knn.fit(X_train, y_train)
          y_pred = knn.predict(X_test)
          result = pd.concat([result,GetMetrics(y_test, y_pred,name)],axis=1)
     return GetMetricsResult(result,name)

In [13]:
# def MatrizConfusion(y_test, y_pred):
#             matriz_confusion = confusion_matrix(y_test, y_pred)

#             # Crear tabla con los valores de la matriz de confusión
#             clases = [0,1]
#             tabla_confusion = pd.DataFrame(matriz_confusion, index=clases, columns=clases)

#             # Imprimir tabla de matriz de confusión
#             print(tabla_confusion)

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
def RandomForest(X_train,y_train,X_test,y_test,name,n_validation = 10):
    result = pd.DataFrame([])
    for i in range(n_validation):
            clf = RandomForestClassifier(n_estimators=100, random_state=42)
#             clf = RandomForestClassifier(max_depth=2, random_state=0)
            clf = clf.fit(X_train,y_train)
            y_pred = clf.predict(X_test)
            result = pd.concat([result,GetMetrics(y_test, y_pred,name)],axis=1)
            print(result)
    return GetMetricsResult(result,name)

In [15]:
from sklearn.linear_model import LogisticRegression
def LogisticRegr(X_train,y_train,X_test,y_test,name,n_validation = 10):
    result = pd.DataFrame([])
    for i in range(n_validation):
      logreg = LogisticRegression(solver='lbfgs', max_iter=100)
      logreg = logreg.fit(X_train,y_train)
      y_pred = logreg.predict(X_test)
      result = pd.concat([result,GetMetrics(y_test, y_pred,name)],axis=1)
    return GetMetricsResult(result,name)

In [16]:
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
def SGD(X_train,y_train,X_test,y_test,name,n_validation = 10):
    result = pd.DataFrame([])
    for i in range(n_validation):
        clf = make_pipeline(StandardScaler(), SGDClassifier())
        clf = clf.fit(X_train,y_train)
        y_pred = clf.predict(X_test)
        result = pd.concat([result,GetMetrics(y_test, y_pred,name)],axis=1)
    return GetMetricsResult(result,name)

In [17]:
from sklearn.tree import DecisionTreeClassifier # Import Decision Tree Classifier
def DecisionTree(X_train,y_train,X_test,y_test,name,n_validation = 10):
    result = pd.DataFrame([])
    for i in range(n_validation):
        clf = DecisionTreeClassifier()
        clf = clf.fit(X_train,y_train)
        y_pred = clf.predict(X_test)
        result = pd.concat([result,GetMetrics(y_test, y_pred,name)],axis=1)
    return GetMetricsResult(result,name)

In [18]:
from sklearn import svm
from sklearn.model_selection import cross_validate
def SVG(X_train,y_train,X_test,y_test,name,n_validation = 10):
    result = pd.DataFrame([])
    for i in range(n_validation):
        clf = svm.SVC(kernel='linear', C=1, random_state=5)
        clf.fit(X_train,y_train)
        y_pred = clf.predict(X_test)
        result = pd.concat([result,GetMetrics(y_test, y_pred,name)],axis=1)
    return GetMetricsResult(result,name)

In [19]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
def ExecuteModel(model,IntanceReduction,X,y,column):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35)
  
  try:
    if IntanceReduction == "RUS":
       rus = RandomUnderSampler(random_state=0)
       X_train, y_train = rus.fit_resample(X_train, y_train)
    elif IntanceReduction == "ROS":
       ros = RandomOverSampler(random_state=0)
       X_train, y_train = ros.fit_resample(X_train, y_train)

    # Algoritmo a ejecutar
    if model == "NB":
      return NaiveBayes(X_train,y_train,X_test,y_test,column,10)
    elif model == "KNN":
      return KNN(X_train,y_train,X_test,y_test,column,10)
    elif model == "DT":
      return DecisionTree(X_train,y_train,X_test,y_test,column,10)
    elif model == "SGD":
      return SGD(X_train,y_train,X_test,y_test,column,10)
    elif model == "LR":
      return LogisticRegr(X_train,y_train,X_test,y_test,column,10)
    elif model == "RF":
      return RandomForest(X_train,y_train,X_test,y_test,column,10)
    elif model == "SVG":
      return SVG(X_train,y_train,X_test,y_test,column,10)
  except Exception as e:
     print(str(e))
     return pd.DataFrame([-1,-1,-1,-1],
                    index=["pr","re","f","Ac"],
                    columns=[column])


In [20]:
import pyodbc 
def insertDb(data):
    conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=JORDANYS;'
                      'Database=magister;'
                      'Trusted_Connection=yes;')
    cursor = conn.cursor()
  
    dataInsert = "'{level_measure}','{project}','{b}','{rank}','{model_selection}','{intance_reduction}','{model}','{precision}','{recall}','{f}','{accuracy}','{count_metric}','{count_row}'".format(
        level_measure = data.level_measure ,
        project = data.project,
        b = data.b,
        rank = data.rank,
        model_selection = data.model_selection,
        intance_reduction = data.intance_reduction,
        model = data.model,
        precision = data.precision,
        recall = data.recall,
        f = data.f,
        accuracy = data.accuracy,
        count_metric = data.count_metric,
        count_row = data.count_row
    )
 
    cursor.execute('''
                INSERT INTO model_results_cross ([level_measure],[project],[b],[rank],[model_selection],[intance_reduction],[model],[precision],[recall],[f],[accuracy],[count_metric],[count_row])
                VALUES
                ('''+ dataInsert+''')

                ''')

    conn.commit()

In [21]:
def SelectMetricsDb(data):
    conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=JORDANYS;'
                      'Database=magister;'
                      'Trusted_Connection=yes;')
    cursor = conn.cursor()
   
    query = "SELECT DISTINCT metric FROM [magister].[dbo].[select_measure] where b = '{b}' and rank = '{rank}' and project = '{project}' and level_measure = '{level_measure}' and cs_su like '%{cs_su}%' and ig_su like '%{ig_su}%' and rf_su like '%{rf_su}%' and cs_cos like '%{cs_cos}%' and ig_cos like '%{ig_cos}%' and rf_cos like '%{rf_cos}%'".format(
        level_measure = data.level_measure,
        project = data.project,
        b = data.b,
        rank = data.rank,
        cs_su = data.cs_su,
        ig_su = data.ig_su,
        rf_su = data.rf_su,
        cs_cos = data.cs_cos,
        ig_cos = data.ig_cos,
        rf_cos = data.rf_cos
    )
    cursor.execute(query)
    results = cursor.fetchall()
    conn.close()
    return [i[0] for i in results] 

In [22]:
def SelectAllMetricsDb():
    conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=JORDANYS;'
                      'Database=magister;'
                      'Trusted_Connection=yes;')
    cursor = conn.cursor()
   
    query = "SELECT * FROM [magister].[dbo].[select_measure]"
    cursor.execute(query)
    df = pd.DataFrame(np.array(cursor.fetchall()),columns =[column[0] for column in cursor.description])
#     columns =[column[0] for column in cursor.description]
    return df

In [23]:
directory = 'C:/Users/danyr/Downloads/Python/full/'
listFile = FileList(directory)
projects =np.unique([os.path.split(os.path.dirname(i))[1] for i in listFile])
print(projects)

['Android-Universal-Image-Loader' 'BroadleafCommerce' 'MapDB' 'all'
 'antlr4' 'ceylon-ide-eclipse' 'elasticsearch' 'hazelcast' 'junit' 'mcMMO'
 'mct' 'neo4j' 'netty' 'orientdb' 'oryx' 'titan']


In [24]:
import csv
def insertFile(path,mode,row):
    with open(path, mode, newline='', encoding='UTF8') as f:
        # create the csv writer
        writer = csv.writer(f)
        # write a row to the csv file
        writer.writerow(row)
        

In [25]:
def insertDbCSV(data,file):
        insertFile(file,'a+',[
            data.level_measure
            ,data.project
            ,data.b
            ,data.rank
            ,data.model_selection
            ,data.intance_reduction
            ,data.model
            ,data.precision
            ,data.recall
            ,data.f
            ,data.accuracy
            ,data.count_metric
            ,data.count_row
            ,data.count_bag
        ]) 

In [26]:
metricsList = SelectAllMetricsDb()


In [27]:
def ModelsByCondition(project):
    file = 'D:/cross4/'+project+'.csv'
    insertFile(file,'w', ['level_measure','project','b','rank','model_selection','intance_reduction','model','precision','recall','f','accuracy','count_metric','count_row','count_bag'])  

    ranks = ['0.5','0.7','0.9']
    bs = ['0.5','0.7','0.9']
    models = ['SGD','LR','RF','NB','KNN','DT']
    models = ['RF','NB','KNN','DT']
    modelsSelections = ['CS_SU','IG_SU','RF_SU','CS_COS','IG_COS','RF_COS','ALL']
    IntanceReductions = ['RUS','ROS','NONE']

 
    for rank in ranks:
          for b in bs:
                paths = [i for i in listFile if os.path.split(os.path.dirname(i))[1] == project] 
                for path in paths:
                    data=pd.read_csv(path)
                    try:
                      data=data.drop(columns=['Project','Hash', 'LongName','Parent'],axis=1, errors='ignore')
                      X =data.drop(labels=['Number of Bugs'], axis=1).astype(float).abs()
                      y =  [1 if i > 0 else 0 for i in data['Number of Bugs']]
                    except Exception as e:
                      print("Error in " + file_location + str(e))
                      continue
                    for model in models:
                        for modelsSelection in modelsSelections:
                            for reduction in IntanceReductions:
                              level = os.path.split(os.path.basename(path))[1].split('.')[0] 
                              dataMetric = metricsList.loc[(metricsList['project'] == project) & (metricsList['level_measure'] == level) & (metricsList['b'] == b) & (metricsList['rank'] == rank)] 
                              
                              if modelsSelection == 'CS_SU':
                                    dataMetric = dataMetric.loc[dataMetric['cs_su'] == 'True']
                              elif modelsSelection == 'IG_SU':
                                    dataMetric = dataMetric.loc[dataMetric['ig_su'] == 'True']
                              elif modelsSelection == 'RF_SU':
                                    dataMetric = dataMetric.loc[dataMetric['rf_su'] == 'True']
                              elif modelsSelection == 'CS_COS':
                                    dataMetric = dataMetric.loc[dataMetric['cs_cos'] == 'True']
                              elif modelsSelection == 'IG_COS':
                                    dataMetric = dataMetric.loc[dataMetric['ig_cos'] == 'True']
                              elif modelsSelection == 'RF_COS':
                                    dataMetric = dataMetric.loc[dataMetric['rf_cos'] == 'True']
                              
                              countBug = 0;
                              for i in y:
                                if i > 0:
                                    countBug = countBug+ 1
                                    
                              if dataMetric.shape[0]>0:
                                  result = ExecuteModel(model,reduction,X[dataMetric['metric']],y,modelsSelection)
                                  insertDbCSV(ModelResult(level,project,b,rank,modelsSelection,reduction,model,result[modelsSelection]['pr'],result[modelsSelection]['re'],result[modelsSelection]['f'],result[modelsSelection]['Ac'],dataMetric.shape[0],X.shape[0],countBug),file)
                                  

In [ ]:
# for project in projects:
#         ModelsByCondition(project)
#         break
from joblib import Parallel, delayed
Parallel(n_jobs=3)(delayed(ModelsByCondition)(project) for project in projects)

In [ ]:
#  ModelsByCondition('oryx')